# La carte du voyage : l'inflation, 2019-2025 · *The map of the journey: inflation, 2019-2025*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_inflation() -> tuple[Series, Series]:
    """Inflation en glissement annuel, É.-U. (IPC, CPIAUCNS) et zone euro (IPCH,
    CP0000EZ19M086NEST), de 2019 à 2025, en direct de FRED.
    Year-over-year inflation, U.S. and euro area, 2019-2025, live from FRED."""
    us_cpi = nm.load_fred("CPIAUCNS", start="2017")
    eu_hicp = nm.load_fred("CP0000EZ19M086NEST", start="2017")
    us = ((us_cpi / us_cpi.shift(12) - 1) * 100).loc["2019":"2025"]
    eu = ((eu_hicp / eu_hicp.shift(12) - 1) * 100).loc["2019":"2025"]
    return us, eu

us, eu = load_inflation()


import matplotlib.dates as mdates
from pandas import Timestamp as T
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="La carte du voyage : l'inflation, 2019-2025",
        legend=["États-Unis (IPC)", "Zone euro (IPCH)"],
        target="cible : 2 %",
        steps=["1 · L'arrêt", "2 · La riposte", "3 · La surchauffe",
               "4 · Le choc & le resserrement", "5 · La désinflation", "6 · L'atterrissage"],
        us_peak="É.-U. : pic à 9,1 % (juin 2022)",
        eu_peak="zone euro : 10,6 %\n(oct. 2022)",
        end_us="2,7 %", end_eu="1,9 %",
        note="Sources : BLS (IPC) et Eurostat (IPCH zone euro), via FRED. Glissement annuel, mensuel, janv. 2019 – déc. 2025.\n"
             "Les six bandes correspondent aux six étapes du chapitre ; valeurs de fin 2025 à droite des courbes."),
    "en": dict(
        title="The map of the journey: inflation, 2019-2025",
        legend=["United States (CPI)", "Euro area (HICP)"],
        target="2% target",
        steps=["1 · The stop", "2 · The response", "3 · The overheat",
               "4 · The shock & tightening", "5 · The disinflation", "6 · The landing"],
        us_peak="U.S.: peak at 9.1% (June 2022)",
        eu_peak="euro area: 10.6%\n(Oct. 2022)",
        end_us="2.7%", end_eu="1.9%",
        note="Sources: BLS (CPI) and Eurostat (euro-area HICP), via FRED. Year-over-year, monthly, Jan. 2019 – Dec. 2025.\n"
             "The six bands mark the chapter's six legs; end-2025 values to the right of the curves."),
}

def build_figure(us: Series, eu: Series, lang: str) -> Figure:
    """Deux courbes d'inflation, six bandes-étapes, pics annotés et valeurs de fin."""
    text = LABELS[lang]
    pct = FuncFormatter(lambda v, _: (f"{v:.0f} %" if lang == "fr" else f"{v:.0f}%").replace("-", "−"))
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.072, top=0.86)

    # Six bandes-étapes alternées (très discrètes) / six alternating step bands.
    bounds = [T("2019-01-01"), T("2020-05-01"), T("2021-03-01"), T("2022-02-01"),
              T("2023-07-01"), T("2024-07-01"), T("2026-01-01")]
    for i in range(6):
        if i % 2 == 1:
            ax.axvspan(bounds[i], bounds[i + 1], color=nm.COLORS["edge"], alpha=0.14, linewidth=0)

    ax.axhline(2, color=nm.COLORS["muted"], linestyle=(0, (6, 4)), linewidth=1.8, alpha=0.85)
    ax.plot(us.index, us, color=nm.COLORS["blue"], linewidth=2.9, label=text["legend"][0])
    ax.plot(eu.index, eu, color=nm.COLORS["rose"], linewidth=2.9, label=text["legend"][1])

    ax.set_ylim(-1, 13.3)
    ax.set_yticks(range(0, 13, 2))
    ax.yaxis.set_major_formatter(pct)
    ax.set_xlim(T("2019-01-01"), T("2026-03-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.text(T("2019-02-01"), 2.55, text["target"], fontsize=20, color=nm.COLORS["muted"], va="bottom")
    step_x = [T("2019-10-01"), T("2020-04-01"), T("2021-02-01"),
              T("2021-11-01"), T("2023-05-01"), T("2024-08-01")]
    for i, (x, label) in enumerate(zip(step_x, text["steps"])):
        ax.text(x, 12.6 if i % 2 == 0 else 11.5, label, fontsize=20.5,
                color=nm.COLORS["muted"], va="center", ha="left")

    us_pk = us.loc["2021":"2023"].idxmax()
    eu_pk = eu.loc["2021":"2023"].idxmax()
    ax.annotate(text["us_peak"], xy=(us_pk, us.loc[us_pk]), xytext=(T("2020-09-01"), 10.4),
                fontsize=21.5, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["eu_peak"], xy=(eu_pk, eu.loc[eu_pk]), xytext=(T("2023-02-01"), 11.2),
                fontsize=21.5, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))

    ax.annotate(text["end_us"], xy=(us.index[-1], us.iloc[-1]), xytext=(10, 6),
                textcoords="offset points", fontsize=26, fontweight="bold",
                color=nm.COLORS["blue2"], va="center", ha="left", annotation_clip=False)
    ax.annotate(text["end_eu"], xy=(eu.index[-1], eu.iloc[-1]), xytext=(10, -8),
                textcoords="offset points", fontsize=26, fontweight="bold",
                color=nm.COLORS["rose"], va="center", ha="left", annotation_clip=False)

    leg = ax.legend(loc="upper left", bbox_to_anchor=(0.015, 0.72), fontsize=22,
                    labelcolor=nm.COLORS["text"], handlelength=1.5, borderpad=0.8,
                    labelspacing=0.6, fancybox=True)
    leg.get_frame().set_facecolor(nm.COLORS["card"])
    leg.get_frame().set_edgecolor(nm.COLORS["edge"])
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(us, eu, LANG)